# 06 — GSM8K SFT từ checkpoint pretrain

    Chạy phiên mới, không chạy lại notebook 04. `RUN_MODE='inspect'` mặc định. `smoke` dùng mô hình/checkpoint pretrain thử, bốn microbatch để kiểm cả validation và snapshot; `production` nạp **trọng số pretrain chính**, tạo optimizer/lịch SFT mới, train 1 epoch trước khi xem validation và quyết định có tiếp tục tối đa 3 epoch. Không dùng Orca. Local: `uv sync --group model --group train --group data --group notebook`; Kaggle cần `tensorboard numpy`, giữ torch CUDA cài sẵn.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import torch

REPO_URL = 'https://github.com/ngochhuy/llm-reasoning.git'
REPO_REF = ''  # Kaggle: đặt commit SHA đã review; không dùng main di động.
ON_KAGGLE = Path('/kaggle/working').exists()
REPO_DIR = Path('/kaggle/working/llm-reasoning-repo') if ON_KAGGLE else next(
    (p for p in (Path.cwd(), Path.cwd().parent) if (p/'src'/'llm_reasoning').exists()), Path.cwd())
if ON_KAGGLE:
    if len(REPO_REF) < 40:
        raise ValueError('Đặt REPO_REF bằng commit SHA 40 ký tự đã review trước khi chạy Kaggle')
    if not (REPO_DIR/'src'/'llm_reasoning').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
print('commit:', CODE_COMMIT, 'dirty:', DIRTY, 'torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
sys.path.insert(0, str(REPO_DIR/'src'))
os.environ['PYTHONPATH'] = str(REPO_DIR/'src') + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
from dataclasses import asdict, replace
from llm_reasoning.checkpoint import file_sha256, load_checkpoint, load_model_artifact, save_checkpoint
from llm_reasoning.config import make_config
from llm_reasoning.data_stream import ResumableBatchStream
from llm_reasoning.tokenizer import ByteBPETokenizer
from llm_reasoning.training import Trainer, TrainingLogger
from llm_reasoning.training_data import SFTBatches
import math, time

RUN_MODE = 'inspect'  # inspect | smoke | production
APPROVED_EARLY_STOP = False  # nếu pretrain dừng theo thời gian, cần review trước SFT.
PRETRAIN_CHECKPOINT = Path('/kaggle/input/llm-reasoning-pretrain/latest.pt') if ON_KAGGLE else REPO_DIR/'artifacts/nb04_pretrain/latest.pt'
TOKENIZER = Path('/kaggle/input/llm-reasoning-data/pretrain/tokenizer.json') if ON_KAGGLE else REPO_DIR/'artifacts/data/pretrain/tokenizer.json'
SFT_ROOT = Path('/kaggle/input/llm-reasoning-data/sft_gsm8k') if ON_KAGGLE else REPO_DIR/'artifacts/data/sft_gsm8k'
OUTPUT_ROOT = Path('/kaggle/working/sft_run') if ON_KAGGLE else REPO_DIR/'artifacts/nb06_sft'
RESUME_FROM = None  # chỉ checkpoint SFT cùng dataset/config/thiết bị
MAX_EPOCHS_THIS_SESSION = 1
if RUN_MODE not in ('inspect', 'smoke', 'production'):
    raise ValueError('RUN_MODE không hợp lệ')
if RUN_MODE == 'production' and (not ON_KAGGLE or not torch.cuda.is_available() or DIRTY):
    raise RuntimeError('Production yêu cầu Kaggle CUDA, commit sạch')
if RESUME_FROM is not None and RUN_MODE != 'production':
    raise ValueError('Chỉ resume cùng SFT production run')
tokenizer = ByteBPETokenizer.load(TOKENIZER)
manifest = json.loads((SFT_ROOT/'manifest.json').read_text(encoding='utf-8'))
if manifest['sft_sources'] != ['gsm8k'] or manifest['tokenizer_sha256'] != file_sha256(TOKENIZER):
    raise ValueError('SFT phải là GSM8K-only và dùng đúng tokenizer pretrain')
_, provenance = load_model_artifact(PRETRAIN_CHECKPOINT, tokenizer_path=TOKENIZER)
if provenance['stage'] != 'pretrain':
    raise ValueError('SFT phải bắt đầu từ checkpoint pretrain')
if RUN_MODE == 'production' and provenance['checkpoint_kind'] != 'training':
    raise ValueError('Không dùng checkpoint smoke làm nguồn production SFT')
if RUN_MODE == 'production' and not APPROVED_EARLY_STOP and provenance['train_state']['attempted_steps'] < provenance['total_steps']:
    raise ValueError('Checkpoint pretrain chưa hoàn tất; chưa bắt đầu SFT')
print('Mode:', RUN_MODE, 'pretrain SHA:', file_sha256(PRETRAIN_CHECKPOINT),
      'SFT rows:', manifest['splits'])

In [ ]:
if RUN_MODE != 'inspect':
    cfg = make_config('smoke' if RUN_MODE == 'smoke' else 'target')
    train_cfg = cfg.train
    if RUN_MODE == 'smoke':
        train_cfg = replace(train_cfg, micro_batch_size=1, gradient_accumulation_steps=1,
                            sft_seq_len=manifest['max_seq_len'], precision='fp32')
    device = 'cpu' if RUN_MODE == 'smoke' else 'cuda'
    train_data = SFTBatches(SFT_ROOT, 'train', batch_size=train_cfg.micro_batch_size,
                             pad_token_id=tokenizer.special_to_id['<pad>'],
                             max_seq_len=train_cfg.sft_seq_len,
                             max_batches=4 if RUN_MODE == 'smoke' else None)
    stream = ResumableBatchStream(train_data, fingerprint=train_data.fingerprint,
                                  seed=cfg.data.seed, shuffle=True)
    total_steps = math.ceil(len(train_data)/train_cfg.gradient_accumulation_steps) * cfg.data.sft_max_epochs
    print('Microbatches/epoch:', len(train_data), 'planned max steps:', total_steps,
          'fingerprint:', train_data.fingerprint)
else:
    print('Inspect hoàn tất; chưa tạo optimizer và chưa train.')

## Tiếp tục và dừng theo validation

    Lần đầu nạp **chỉ model weights** từ pretrain; optimizer, scheduler, scaler SFT mới. Resume từ `latest.pt` SFT mới khôi phục đủ trạng thái. Bắt đầu 1 epoch; xem validation rồi quyết định có tiếp tục tối đa 3 epoch. Ngân sách 8M token được kiểm ở biên update nên có thể vượt tối đa một accumulation window. Giữ snapshot mỗi epoch để chọn checkpoint theo validation sau review; `latest.pt` phục vụ resume. Copy output vào nguồn lưu bền vững sau phiên.

In [ ]:
if RUN_MODE != 'inspect':
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    latest = OUTPUT_ROOT/'latest.pt'
    log_dir = OUTPUT_ROOT/'logs'
    if RESUME_FROM is None:
        if latest.exists() or (log_dir/'metrics.csv').exists():
            raise FileExistsError('Output cũ tồn tại; chọn thư mục mới hoặc RESUME_FROM')
        torch.manual_seed(cfg.data.seed)
        model, _ = load_model_artifact(PRETRAIN_CHECKPOINT, tokenizer_path=TOKENIZER, device=device)
        trainer = Trainer(model, train_cfg, stage='sft', total_steps=total_steps)
    else:
        trainer = load_checkpoint(RESUME_FROM, stream, tokenizer_path=TOKENIZER,
                                  device=device, expected_stage='sft')
        if trainer.total_steps != total_steps or trainer.config != train_cfg:
            raise ValueError('Config SFT/total_steps thay đổi sau resume')
    kind = 'smoke' if RUN_MODE == 'smoke' else 'training'
    run_manifest = dict(stage='sft', kind=kind, code_commit=CODE_COMMIT,
                        pretrained_checkpoint_sha256=file_sha256(PRETRAIN_CHECKPOINT),
                        tokenizer_sha256=file_sha256(TOKENIZER),
                        data_fingerprint=train_data.fingerprint,
                        train_config=asdict(train_cfg), total_steps=total_steps, device=device)
    run_manifest_path = OUTPUT_ROOT/'run_manifest.json'
    if run_manifest_path.exists():
        if json.loads(run_manifest_path.read_text(encoding='utf-8')) != run_manifest:
            raise ValueError('Run manifest SFT thay đổi; không thể resume cùng output')
    else:
        run_manifest_path.write_text(json.dumps(run_manifest, indent=2)+'\n', encoding='utf-8')
    with TrainingLogger(log_dir, resume_step=trainer.state.attempted_steps if RESUME_FROM else None) as logger:
        trainer.logger = logger
        epochs_completed = 0
        last_saved = time.monotonic()
        while epochs_completed < MAX_EPOCHS_THIS_SESSION and stream.epoch < cfg.data.sft_max_epochs \
                and trainer.state.attempted_steps < trainer.total_steps \
                and trainer.state.processed_tokens < cfg.data.sft_max_tokens:
            rows = trainer.train_epoch(stream, max_updates=1)
            if rows and time.monotonic() - last_saved >= train_cfg.checkpoint_interval_minutes * 60:
                save_checkpoint(latest, trainer, stream, tokenizer_path=TOKENIZER, checkpoint_kind=kind)
                last_saved = time.monotonic()
            if stream.cursor == len(stream.batches):
                validation = SFTBatches(SFT_ROOT, 'validation', batch_size=train_cfg.micro_batch_size,
                                        pad_token_id=tokenizer.special_to_id['<pad>'],
                                        max_seq_len=train_cfg.sft_seq_len,
                                        max_batches=2 if RUN_MODE == 'smoke' else None)
                score = trainer.evaluate(validation)
                snapshot = OUTPUT_ROOT/f'epoch_{stream.epoch + 1}.pt'
                save_checkpoint(snapshot, trainer, stream, tokenizer_path=TOKENIZER, checkpoint_kind=kind)
                print('Epoch', stream.epoch + 1, 'validation loss', score['loss'], 'snapshot', snapshot)
                epochs_completed += 1
                if stream.epoch + 1 < cfg.data.sft_max_epochs and trainer.state.processed_tokens < cfg.data.sft_max_tokens:
                    stream.next_epoch()
            elif not rows:
                raise RuntimeError('Stream chưa hết nhưng không có update; kiểm tra total_steps')
        save_checkpoint(latest, trainer, stream, tokenizer_path=TOKENIZER, checkpoint_kind=kind)
    print('SFT updates:', trainer.state.attempted_steps, 'tokens:', trainer.state.processed_tokens,
          'epoch:', stream.epoch, 'cursor:', stream.cursor, 'latest:', latest)